In [20]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D10 — IMPI Questionnaire
# ============================================================

!pip install -q pymupdf

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import platform
import re
import sys

import fitz
import pandas as pd

In [21]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D10"

DOCUMENT_NAME = (
    "IMPI — Inquérito Mensal à Produção Industrial"
)

SOURCE_FORMAT = "PDF"

EXPECTED_PAGE_COUNT = 4
EXPECTED_REFERENCE_RECORD_COUNT = 69

EXPECTED_CATEGORY_COUNTS = {
    "Instrument metadata": 8,
    "Questionnaire field": 32,
    "UAE template element": 6,
    "Product table field": 12,
    "Instruction": 11
}

REFERENCE_FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Code",
    "Expected Value Type",
    "Source Location"
]

ALLOWED_CATEGORIES = set(
    EXPECTED_CATEGORY_COUNTS
)

OUTPUT_DIR = Path("outputs_D10_stage1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_DIAGNOSTICS_PATH = (
    OUTPUT_DIR / "D10_source_diagnostics.json"
)

DOCUMENT_CHARACTERISATION_PATH = (
    OUTPUT_DIR / "D10_document_characterisation.json"
)

PAGE_CHARACTERISATION_PATH = (
    OUTPUT_DIR / "D10_page_characterisation.csv"
)

REFERENCE_VALUES_PATH = (
    OUTPUT_DIR / "D10_reference_values.csv"
)

REFERENCE_SCHEMA_PATH = (
    OUTPUT_DIR / "D10_reference_schema.json"
)

REFERENCE_SUMMARY_PATH = (
    OUTPUT_DIR / "D10_reference_summary.json"
)

REFERENCE_METADATA_PATH = (
    OUTPUT_DIR / "D10_reference_metadata.json"
)

REFERENCE_INTEGRITY_PATH = (
    OUTPUT_DIR / "D10_reference_integrity.json"
)


REFERENCE_VALUES_JSON_PATH = (
    OUTPUT_DIR / "D10_reference_values.json"
)

EXTRACTION_TASK_PATH = (
    OUTPUT_DIR / "D10_extraction_task.txt"
)

EXTRACTION_SCHEMA_PATH = (
    OUTPUT_DIR / "D10_extraction_schema.json"
)

QUALITY_EVIDENCE_PATH = (
    OUTPUT_DIR / "D10_quality_evidence.json"
)

INDICATOR_ASSESSMENT_PATH = (
    OUTPUT_DIR / "D10_indicator_assessment.csv"
)

DIMENSION_ASSESSMENT_PATH = (
    OUTPUT_DIR / "D10_dimension_assessment.csv"
)

print("Document:", DOCUMENT_ID)
print("Expected pages:", EXPECTED_PAGE_COUNT)
print("Expected reference records:", EXPECTED_REFERENCE_RECORD_COUNT)
print("Output directory:", OUTPUT_DIR)


Document: D10
Expected pages: 4
Expected reference records: 69
Output directory: outputs_D10_stage1


In [22]:
# ============================================================
# 2. Upload source PDF
# ============================================================

print(
    "Upload the D10 IMPI questionnaire PDF."
)

uploaded = files.upload()

pdf_paths = [
    Path(filename)
    for filename in uploaded
    if filename.lower().endswith(".pdf")
]

if len(pdf_paths) != 1:
    raise ValueError(
        "Upload exactly one PDF source document."
    )

SOURCE_PATH = pdf_paths[0]

print("Source file:", SOURCE_PATH.name)
print("Source size:", f"{SOURCE_PATH.stat().st_size:,} bytes")


Upload the D10 IMPI questionnaire PDF.


Saving D10 - IMPI_QUESTIONARIO INE Portugal 2026.pdf to D10 - IMPI_QUESTIONARIO INE Portugal 2026 (1).pdf
Source file: D10 - IMPI_QUESTIONARIO INE Portugal 2026 (1).pdf
Source size: 278,787 bytes


In [23]:
# ============================================================
# 3. Hashing utility
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

print("Source SHA-256:", SOURCE_SHA256)


Source SHA-256: fb72aac548f61578bc9ba52448793b81bf61e37eff146f9519124d0794a4f4e5


In [24]:
# ============================================================
# 4. Load PDF and inspect source structure
# ============================================================

pdf_document = fitz.open(
    SOURCE_PATH
)

PAGE_COUNT = len(
    pdf_document
)

PAGE_COUNT_VALID = (
    PAGE_COUNT == EXPECTED_PAGE_COUNT
)

page_rows = []
page_texts = []

for page_number, page in enumerate(
    pdf_document,
    start=1
):
    text = page.get_text("text") or ""
    blocks = page.get_text("blocks")
    drawings = page.get_drawings()
    images = page.get_images(full=True)
    widgets = list(page.widgets() or [])

    page_texts.append(
        {
            "page_number": page_number,
            "text": text
        }
    )

    page_rows.append(
        {
            "Page Number": page_number,
            "Text Characters": len(text),
            "Word Count": len(text.split()),
            "Non-empty Lines": len(
                [
                    line
                    for line in text.splitlines()
                    if line.strip()
                ]
            ),
            "Text Block Count": len(blocks),
            "Drawing Count": len(drawings),
            "Embedded Image Count": len(images),
            "Interactive Widget Count": len(widgets),
            "Width": float(page.rect.width),
            "Height": float(page.rect.height),
            "Orientation": (
                "landscape"
                if page.rect.width > page.rect.height
                else "portrait"
            )
        }
    )


page_characterisation_df = pd.DataFrame(
    page_rows
)

FULL_TEXT = "\n".join(
    page["text"]
    for page in page_texts
)

TEXT_EXTRACTABLE = bool(
    FULL_TEXT.strip()
)

OCR_REQUIRED = not TEXT_EXTRACTABLE

print("Page count:", PAGE_COUNT)
print("Page count valid:", PAGE_COUNT_VALID)
print("Text characters:", len(FULL_TEXT))
print("Text extractable:", TEXT_EXTRACTABLE)
print("OCR required:", OCR_REQUIRED)

display(page_characterisation_df)

if not PAGE_COUNT_VALID:
    raise AssertionError(
        f"Expected {EXPECTED_PAGE_COUNT} pages, "
        f"found {PAGE_COUNT}."
    )


Page count: 4
Page count valid: True
Text characters: 6676
Text extractable: True
OCR required: False


,Page Number,Text Characters,Word Count,Non-empty Lines,Text Block Count,Drawing Count,Embedded Image Count,Interactive Widget Count,Width,Height,Orientation
0,1,1633,242,69,36,215,2,0,595.200012,841.679993,portrait
1,2,748,111,19,13,54,1,0,595.200012,841.679993,portrait
2,3,327,49,31,16,1044,0,0,595.200012,841.679993,portrait
3,4,3965,587,53,15,25,0,0,595.200012,841.679993,portrait


In [25]:
# ============================================================
# 5. Source-text diagnostics and marker validation
# ============================================================

numeric_tokens = re.findall(
    r"(?<!\w)-?\d+(?:[.,]\d+)?",
    FULL_TEXT
)

code_tokens = re.findall(
    r"\bBC\d{3}\b",
    FULL_TEXT
)

url_tokens = re.findall(
    r"https?://[^\s]+",
    FULL_TEXT
)

email_tokens = re.findall(
    r"[\w.%-]+@[\w.-]+\.[A-Za-z]{2,}",
    FULL_TEXT
)


SOURCE_MARKER_PATTERNS = {
    "survey_title":
        r"IMPI\s*-\s*Inquérito Mensal à Produção Industrial",

    "statistical_system":
        r"INSTRUMENTO DE NOTAÇÃO DO SISTEMA ESTATÍSTICO NACIONAL",

    "legal_basis":
        r"LEI\s+N[º°]\s*22/2008\s+DE\s+13\s+DE\s+MAIO",

    "registration_number":
        r"N[º°]\s*10067",

    "validity_date":
        r"2026/12/31",

    "response_url":
        r"webInq\.ine\.pt/aderentes",

    "contact_email":
        r"ipi@ine\.pt",

    "identification_section":
        r"Identificação da unidade estatística",

    "activity_section":
        r"Situação da unidade estatística",

    "uae_section":
        r"Unidade de Atividade Económica",

    "product_table":
        r"QUANTIDADES\s+PRODUZIDAS",

    "filling_instructions":
        r"INSTRUÇÕES DE PREENCHIMENTO",

    "explanatory_notes":
        r"NOTAS EXPLICATIVAS",

    "monetary_example":
        r"6370,65",

    "prodcom_reference":
        r"3924/91",

    "snc_accounts":
        r"Contas?\s+SNC\s+712\s+e\s+713"
}


source_marker_status = {
    marker: bool(
        re.search(
            pattern,
            FULL_TEXT,
            flags=re.IGNORECASE
        )
    )
    for marker, pattern
    in SOURCE_MARKER_PATTERNS.items()
}

source_markers_valid = all(
    source_marker_status.values()
)


SOURCE_DIAGNOSTICS = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "source_format": SOURCE_FORMAT,
    "expected_page_count": EXPECTED_PAGE_COUNT,
    "observed_page_count": PAGE_COUNT,
    "page_count_valid": PAGE_COUNT_VALID,
    "text_extractable": TEXT_EXTRACTABLE,
    "ocr_required": OCR_REQUIRED,
    "character_count": len(FULL_TEXT),
    "word_count": len(FULL_TEXT.split()),
    "non_empty_line_count": len(
        [
            line
            for line in FULL_TEXT.splitlines()
            if line.strip()
        ]
    ),
    "numeric_token_count": len(numeric_tokens),
    "questionnaire_code_count": len(code_tokens),
    "unique_questionnaire_codes": sorted(
        set(code_tokens)
    ),
    "url_count": len(url_tokens),
    "email_count": len(email_tokens),
    "source_marker_status": source_marker_status,
    "source_markers_valid": source_markers_valid,
    "interactive_pdf_widgets_detected": int(
        page_characterisation_df[
            "Interactive Widget Count"
        ].sum()
    ),
    "notes": (
        "The PDF contains extractable text but interpretation also "
        "depends on visual form layout, checkboxes, coded labels, "
        "repeated UAE blocks and a grid-like product table."
    )
}


SOURCE_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        SOURCE_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

print(
    json.dumps(
        SOURCE_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

if not source_markers_valid:
    raise AssertionError(
        "One or more required D10 source-content markers were not found."
    )


{
  "document_id": "D10",
  "document_name": "IMPI — Inquérito Mensal à Produção Industrial",
  "source_file": "D10 - IMPI_QUESTIONARIO INE Portugal 2026 (1).pdf",
  "source_file_sha256": "fb72aac548f61578bc9ba52448793b81bf61e37eff146f9519124d0794a4f4e5",
  "source_format": "PDF",
  "expected_page_count": 4,
  "observed_page_count": 4,
  "page_count_valid": true,
  "text_extractable": true,
  "ocr_required": false,
  "character_count": 6676,
  "word_count": 989,
  "non_empty_line_count": 172,
  "numeric_token_count": 38,
  "questionnaire_code_count": 8,
  "unique_questionnaire_codes": [
    "BC001",
    "BC005",
    "BC007",
    "BC010",
    "BC015",
    "BC020",
    "BC025",
    "BC030"
  ],
  "url_count": 1,
  "email_count": 1,
  "source_marker_status": {
    "survey_title": true,
    "statistical_system": true,
    "legal_basis": true,
    "registration_number": true,
    "validity_date": true,
    "response_url": true,
    "contact_email": true,
    "identification_section": true,


In [26]:
# ============================================================
# 6. Fixed extraction task
# ============================================================

EXTRACTION_TASK = """
You are an information extraction assistant.

Extract the fixed set of structural and semantic records represented
in the supplied D10 IMPI — Inquérito Mensal à Produção Industrial
questionnaire.

Treat the supplied document as the only source.

Return exactly 69 records:

- 8 Instrument metadata
- 32 Questionnaire field
- 6 UAE template element
- 12 Product table field
- 11 Instruction

For every record return exactly:

- Category
- Section
- Field or Concept
- Code
- Expected Value Type
- Description
- Source Location

Rules:

- Extract only information explicitly represented in the source.
- Preserve Portuguese source wording where applicable.
- Preserve printed questionnaire codes such as BC001, BC005 and BC030.
- Treat blank response areas as fields, not as observed respondent values.
- Treat the three UAE blocks on page 2 as repeated instances of one
  template; do not create duplicate records for each repeated block.
- Treat the sample product rows on page 3 as examples rather than
  respondent observations.
- Preserve the product-table column structure.
- Preserve explanatory definitions and instructions included in the
  fixed reference scope.
- Use null for Code when no printed questionnaire code is associated
  with the represented element.
- Do not calculate, infer, derive, repair or introduce information not
  explicitly represented in the source.
- Return exactly 69 records.
- Return valid JSON using the exact field names defined in the
  extraction schema.
- Do not include explanations before or after the JSON.
"""

print(EXTRACTION_TASK)



You are an information extraction assistant.

Extract the fixed set of structural and semantic records represented
in the supplied D10 IMPI — Inquérito Mensal à Produção Industrial
questionnaire.

Treat the supplied document as the only source.

Return exactly 69 records:

- 8 Instrument metadata
- 32 Questionnaire field
- 6 UAE template element
- 12 Product table field
- 11 Instruction

For every record return exactly:

- Category
- Section
- Field or Concept
- Code
- Expected Value Type
- Description
- Source Location

Rules:

- Extract only information explicitly represented in the source.
- Preserve Portuguese source wording where applicable.
- Preserve printed questionnaire codes such as BC001, BC005 and BC030.
- Treat blank response areas as fields, not as observed respondent values.
- Treat the three UAE blocks on page 2 as repeated instances of one
  template; do not create duplicate records for each repeated block.
- Treat the sample product rows on page 3 as examples rather 

In [27]:
# ============================================================
# 7. Extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id": DOCUMENT_ID,

    "record_level":
        "questionnaire_structural_or_instruction_record",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "fields": {
        "Category": {
            "type": ["string", "null"]
        },

        "Section": {
            "type": ["string", "null"]
        },

        "Field or Concept": {
            "type": ["string", "null"]
        },

        "Code": {
            "type": ["string", "null"]
        },

        "Expected Value Type": {
            "type": ["string", "null"]
        },

        "Description": {
            "type": ["string", "null"]
        },

        "Source Location": {
            "type": ["string", "null"]
        }
    },

    "expected_output_structure": {
        "document_id": DOCUMENT_ID,

        "records": [
            {
                "Category": "string or null",
                "Section": "string or null",
                "Field or Concept": "string or null",
                "Code": "string or null",
                "Expected Value Type": "string or null",
                "Description": "string or null",
                "Source Location": "string or null"
            }
        ]
    }
}

print(
    json.dumps(
        EXTRACTION_SCHEMA,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D10",
  "record_level": "questionnaire_structural_or_instruction_record",
  "expected_record_count": 69,
  "expected_category_counts": {
    "Instrument metadata": 8,
    "Questionnaire field": 32,
    "UAE template element": 6,
    "Product table field": 12,
    "Instruction": 11
  },
  "fields": {
    "Category": {
      "type": [
        "string",
        "null"
      ]
    },
    "Section": {
      "type": [
        "string",
        "null"
      ]
    },
    "Field or Concept": {
      "type": [
        "string",
        "null"
      ]
    },
    "Code": {
      "type": [
        "string",
        "null"
      ]
    },
    "Expected Value Type": {
      "type": [
        "string",
        "null"
      ]
    },
    "Description": {
      "type": [
        "string",
        "null"
      ]
    },
    "Source Location": {
      "type": [
        "string",
        "null"
      ]
    }
  },
  "expected_output_structure": {
    "document_id": "D10",
    "records": [
 

In [28]:
# ============================================================
# 8. Reference-value schema
# ============================================================

REFERENCE_SCHEMA = {
    "document_id": DOCUMENT_ID,

    "record_level":
        "questionnaire_structural_or_instruction_record",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "fields": {
        "Category":
            "Fixed reference-record category.",

        "Section":
            "Source section or structural region containing the element.",

        "Field or Concept":
            "Explicit source label, heading, field name or represented element.",

        "Code":
            "Printed questionnaire code associated with the element, "
            "or null when no code is represented.",

        "Expected Value Type":
            "Document-grounded structural annotation describing the "
            "expected response or element type.",

        "Description":
            "Source-grounded description of the represented field, "
            "structural element or instruction.",

        "Source Location":
            "Physical PDF page and source section or element."
    },

    "null_policy": (
        "Code is null when no printed questionnaire code is associated "
        "with the represented element. Blank questionnaire response "
        "areas are represented as fields rather than as null respondent "
        "observations."
    ),

    "reference_construction_rules": [
        (
            "The three UAE blocks on PDF page 2 are treated as repeated "
            "instances of one structural template."
        ),
        (
            "Repeated UAE labels are therefore represented once in the "
            "reference dataset rather than duplicated three times."
        ),
        (
            "Sample product rows on PDF page 3 are treated as examples "
            "and not as observed respondent data."
        ),
        (
            "Expected Value Type is a manual document-grounded structural "
            "annotation rather than an observed respondent value."
        ),
        (
            "Only information explicitly represented in the supplied "
            "document is included."
        )
    ],

    "branch_reuse": (
        "The same fixed reference dataset is reused for "
        "Branches A, B and C."
    )
}

In [29]:
# ============================================================
# 9. Fixed reference values
# ============================================================

reference_values = [
    # --------------------------------------------------------
    # Instrument metadata — 8
    # --------------------------------------------------------
    {
        "Category": "Instrument metadata",
        "Section": "Header",
        "Field or Concept": "Survey name",
        "Description": "IMPI - Inquérito Mensal à Produção Industrial",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 1 — Header"
    },
    {
        "Category": "Instrument metadata",
        "Section": "Header",
        "Field or Concept": "Statistical system",
        "Description": "Instrumento de notação do Sistema Estatístico Nacional",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 1 — Legal notice"
    },
    {
        "Category": "Instrument metadata",
        "Section": "Header",
        "Field or Concept": "Legal basis",
        "Description": "Lei nº 22/2008 de 13 de Maio; resposta confidencial e obrigatória",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 1 — Legal notice"
    },
    {
        "Category": "Instrument metadata",
        "Section": "Header",
        "Field or Concept": "INE registration number",
        "Description": "Registado no INE sob o nº 10067",
        "Code": None,
        "Expected Value Type": "identifier",
        "Source Location": "PDF page 1 — Legal notice"
    },
    {
        "Category": "Instrument metadata",
        "Section": "Header",
        "Field or Concept": "Validity date",
        "Description": "Válido até 2026/12/31",
        "Code": None,
        "Expected Value Type": "date",
        "Source Location": "PDF page 1 — Legal notice"
    },
    {
        "Category": "Instrument metadata",
        "Section": "Response contacts",
        "Field or Concept": "Electronic response URL",
        "Description": "https://webInq.ine.pt/aderentes",
        "Code": None,
        "Expected Value Type": "URL",
        "Source Location": "PDF page 1 — Response information"
    },
    {
        "Category": "Instrument metadata",
        "Section": "Response contacts",
        "Field or Concept": "Contact email",
        "Description": "ipi@ine.pt",
        "Code": None,
        "Expected Value Type": "email",
        "Source Location": "PDF page 1 — Response contacts"
    },
    {
        "Category": "Instrument metadata",
        "Section": "Response contacts",
        "Field or Concept": "Contact telephone",
        "Description": "218 440 479",
        "Code": None,
        "Expected Value Type": "telephone number",
        "Source Location": "PDF page 1 — Response contacts"
    },

    # --------------------------------------------------------
    # Questionnaire field — 32
    # --------------------------------------------------------
     {
        "Category": "Questionnaire field",
        "Section": "Reference data",
        "Field or Concept": "Referência dos dados",
        "Description": (
            "Reference period associated with the questionnaire response"
        ),
        "Code": None,
        "Expected Value Type": "reference period",
        "Source Location": "PDF page 1 — Reference data"
    },

    {
        "Category": "Questionnaire field",
        "Section": "Reference data",
        "Field or Concept": "NIF",
        "Description": "Número de identificação fiscal shown in the reference-data box",
        "Code": None,
        "Expected Value Type": "numeric identifier",
        "Source Location": "PDF page 1 — Reference data"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Identification of statistical unit",
        "Field or Concept": "Número de identificação fiscal (NIF)",
        "Description": "Fiscal identification number of the statistical unit",
        "Code": None,
        "Expected Value Type": "numeric identifier",
        "Source Location": "PDF page 1 — Section I"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Identification of statistical unit",
        "Field or Concept": "Homepage",
        "Description": "Homepage of the statistical unit",
        "Code": None,
        "Expected Value Type": "URL or text",
        "Source Location": "PDF page 1 — Section I"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Identification of statistical unit",
        "Field or Concept": "Designação social",
        "Description": "Legal or social designation of the statistical unit",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 1 — Section I"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Identification of statistical unit",
        "Field or Concept": "Distrito/Ilha",
        "Description": "District or island",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 1 — Section I"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Identification of statistical unit",
        "Field or Concept": "Município",
        "Description": "Municipality",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 1 — Section I"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Identification of statistical unit",
        "Field or Concept": "Freguesia",
        "Description": "Civil parish",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 1 — Section I"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Identification of statistical unit",
        "Field or Concept": "Endereço",
        "Description": "Address",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 1 — Section I"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Identification of statistical unit",
        "Field or Concept": "Localidade",
        "Description": "Locality",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 1 — Section I"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Identification of statistical unit",
        "Field or Concept": "Código postal",
        "Description": "Postal code",
        "Code": None,
        "Expected Value Type": "postal code",
        "Source Location": "PDF page 1 — Section I"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Identification of statistical unit",
        "Field or Concept": "Telefone",
        "Description": "Telephone number of the statistical unit",
        "Code": None,
        "Expected Value Type": "telephone number",
        "Source Location": "PDF page 1 — Section I"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Identification of statistical unit",
        "Field or Concept": "Fax",
        "Description": "Fax number of the statistical unit",
        "Code": None,
        "Expected Value Type": "telephone number",
        "Source Location": "PDF page 1 — Section I"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Identification of statistical unit",
        "Field or Concept": "e-mail",
        "Description": "Email address of the statistical unit",
        "Code": None,
        "Expected Value Type": "email",
        "Source Location": "PDF page 1 — Section I"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Activity status",
        "Field or Concept": "Situação na atividade",
        "Description": "Situation of the statistical unit in the reference period",
        "Code": "BC005",
        "Expected Value Type": "category",
        "Source Location": "PDF page 1 — Section II"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Activity status",
        "Field or Concept": "Aguarda início de atividade",
        "Description": "Unit awaiting the start of activity",
        "Code": None,
        "Expected Value Type": "checkbox",
        "Source Location": "PDF page 1 — Section II"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Activity status",
        "Field or Concept": "Em atividade",
        "Description": "Unit in activity",
        "Code": None,
        "Expected Value Type": "checkbox",
        "Source Location": "PDF page 1 — Section II"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Activity status",
        "Field or Concept": "Atividade suspensa em",
        "Description": "Date on which activity was suspended",
        "Code": "BC010",
        "Expected Value Type": "date",
        "Source Location": "PDF page 1 — Section II"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Activity status",
        "Field or Concept": "Atividade cessada em",
        "Description": "Date on which activity ceased",
        "Code": None,
        "Expected Value Type": "date",
        "Source Location": "PDF page 1 — Section II"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Activity status",
        "Field or Concept": "Nº dias de atividade no período de referência",
        "Description": "Number of activity days in the reference period",
        "Code": "BC007",
        "Expected Value Type": "integer",
        "Source Location": "PDF page 1 — Section II"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Activity status",
        "Field or Concept": "Atividade económica principal (CAE Rev. 3)",
        "Description": "Main economic activity according to CAE Rev. 3",
        "Code": "BC001",
        "Expected Value Type": "CAE code",
        "Source Location": "PDF page 1 — Section II"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Activity status",
        "Field or Concept": "Ocorreu algum facto relevante no período de referência dos dados?",
        "Description": "Whether a relevant fact occurred in the reference period",
        "Code": "BC015",
        "Expected Value Type": "yes or no",
        "Source Location": "PDF page 1 — Section II"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Activity status",
        "Field or Concept": "Indique qual",
        "Description": "Description of the relevant fact",
        "Code": "BC025",
        "Expected Value Type": "text",
        "Source Location": "PDF page 1 — Section II"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Activity status",
        "Field or Concept": "Data",
        "Description": "Date associated with the relevant fact",
        "Code": "BC020",
        "Expected Value Type": "date",
        "Source Location": "PDF page 1 — Section II"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Observations",
        "Field or Concept": "Observações",
        "Description": "Suggestions, justifications or other observations",
        "Code": "BC030",
        "Expected Value Type": "free text",
        "Source Location": "PDF page 1 — Section III"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Responsible person",
        "Field or Concept": "Nome contacto",
        "Description": "Name of the person responsible for completion",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 1 — Section IV"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Responsible person",
        "Field or Concept": "Telefone",
        "Description": "Telephone number of the responsible person",
        "Code": None,
        "Expected Value Type": "telephone number",
        "Source Location": "PDF page 1 — Section IV"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Responsible person",
        "Field or Concept": "Fax",
        "Description": "Fax number of the responsible person",
        "Code": None,
        "Expected Value Type": "telephone number",
        "Source Location": "PDF page 1 — Section IV"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Responsible person",
        "Field or Concept": "e-mail",
        "Description": "Email address of the responsible person",
        "Code": None,
        "Expected Value Type": "email",
        "Source Location": "PDF page 1 — Section IV"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Responsible person",
        "Field or Concept": "Função",
        "Description": "Function of the responsible person",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 1 — Section IV"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Responsible person",
        "Field or Concept": "Assinatura",
        "Description": "Signature of the responsible person",
        "Code": None,
        "Expected Value Type": "signature",
        "Source Location": "PDF page 1 — Section IV"
    },
    {
        "Category": "Questionnaire field",
        "Section": "Responsible person",
        "Field or Concept": "Data",
        "Description": "Date of questionnaire completion",
        "Code": None,
        "Expected Value Type": "date",
        "Source Location": "PDF page 1 — Section IV"
    },

    # --------------------------------------------------------
    # Product table field — 12
    # --------------------------------------------------------
    {
        "Category": "Product table field",
        "Section": "Product production table",
        "Field or Concept": "NIF",
        "Description": "Fiscal identification number",
        "Code": None,
        "Expected Value Type": "numeric identifier",
        "Source Location": "PDF page 3 — Table header"
    },
    {
        "Category": "Product table field",
        "Section": "Product production table",
        "Field or Concept": "UAE",
        "Description": "Economic activity unit",
        "Code": None,
        "Expected Value Type": "identifier",
        "Source Location": "PDF page 3 — Table header"
    },
    {
        "Category": "Product table field",
        "Section": "Product production table",
        "Field or Concept": "Período de Referência",
        "Description": "Reference period of the reported production data",
        "Code": None,
        "Expected Value Type": "month or period",
        "Source Location": "PDF page 3 — Table header"
    },
    {
        "Category": "Product table field",
        "Section": "Product production table",
        "Field or Concept": "Nº",
        "Description": "Line number of the product record",
        "Code": None,
        "Expected Value Type": "integer",
        "Source Location": "PDF page 3 — Product table"
    },
    {
        "Category": "Product table field",
        "Section": "Product production table",
        "Field or Concept": "Produto",
        "Description": "Product designation",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 3 — Product table"
    },
    {
        "Category": "Product table field",
        "Section": "Product production table",
        "Field or Concept": "Unid.",
        "Description": "Reference unit of the product",
        "Code": None,
        "Expected Value Type": "unit",
        "Source Location": "PDF page 3 — Product table"
    },
    {
        "Category": "Product table field",
        "Section": "Product production table",
        "Field or Concept": "Código",
        "Description": "Product code",
        "Code": None,
        "Expected Value Type": "product code",
        "Source Location": "PDF page 3 — Product table"
    },
    {
        "Category": "Product table field",
        "Section": "Product production table",
        "Field or Concept": "Quantidades produzidas",
        "Description": "Quantity produced in the reference period",
        "Code": None,
        "Expected Value Type": "numeric quantity",
        "Source Location": "PDF page 3 — Product table"
    },
    {
        "Category": "Product table field",
        "Section": "Product production table",
        "Field or Concept": "Quantidades vendidas",
        "Description": "Quantity sold in the reference period",
        "Code": None,
        "Expected Value Type": "numeric quantity",
        "Source Location": "PDF page 3 — Product table"
    },
    {
        "Category": "Product table field",
        "Section": "Product production table",
        "Field or Concept": "Valor das vendas / prestação de serviços",
        "Description": "Sales value or value of industrial services",
        "Code": None,
        "Expected Value Type": "monetary value",
        "Source Location": "PDF page 3 — Product table"
    },
    {
        "Category": "Product table field",
        "Section": "Product production table",
        "Field or Concept": "Observações empresa",
        "Description": "Observations provided by the company",
        "Code": None,
        "Expected Value Type": "free text",
        "Source Location": "PDF page 3 — Product table"
    },
    {
        "Category": "Product table field",
        "Section": "Product production table",
        "Field or Concept": "Observações INE",
        "Description": "INE observations; column available only in WebReg",
        "Code": None,
        "Expected Value Type": "free text",
        "Source Location": "PDF page 3 — Product table"
    },

    # --------------------------------------------------------
    # Instruction — 17
    # --------------------------------------------------------
    {
        "Category": "Instruction",
        "Section": "UAE information",
        "Field or Concept": "Unidade de Atividade Económica (UAE)",
        "Description": "Production in the UAE includes own-raw-material production, work on behalf of others and production for intraconsumption in a different UAE",
        "Code": None,
        "Expected Value Type": "instruction",
        "Source Location": "PDF page 2 — Introductory instruction"
    },
    {
        "Category": "UAE template element",
        "Section": "UAE information",
        "Field or Concept": "Código da UAE",
        "Description": "Code placeholder for the economic activity unit",
        "Code": None,
        "Expected Value Type": "identifier",
        "Source Location": "PDF page 2 — Repeated UAE block"
    },
    {
        "Category": "UAE template element",
        "Section": "UAE information",
        "Field or Concept": "Designação da UAE",
        "Description": "Designation placeholder for the economic activity unit",
        "Code": None,
        "Expected Value Type": "text",
        "Source Location": "PDF page 2 — Repeated UAE block"
    },
    {
        "Category": "UAE template element",
        "Section": "UAE information",
        "Field or Concept": "Situação da UAE perante a atividade",
        "Description": "Activity-status selection for the UAE",
        "Code": None,
        "Expected Value Type": "category",
        "Source Location": "PDF page 2 — Repeated UAE block"
    },
    {
        "Category": "UAE template element",
        "Section": "UAE information",
        "Field or Concept": "Observações da UAE",
        "Description": "Observations concerning the UAE",
        "Code": None,
        "Expected Value Type": "free text",
        "Source Location": "PDF page 2 — Repeated UAE block"
    },
    {
        "Category": "UAE template element",
        "Section": "UAE information",
        "Field or Concept": "Confirmar",
        "Description": "Action to confirm the UAE activity status",
        "Code": None,
        "Expected Value Type": "button or action",
        "Source Location": "PDF page 2 — Repeated UAE block"
    },
    {
        "Category": "UAE template element",
        "Section": "UAE information",
        "Field or Concept": "Produtos",
        "Description": "Action to access or confirm products associated with the UAE",
        "Code": None,
        "Expected Value Type": "button or action",
        "Source Location": "PDF page 2 — Repeated UAE block"
    },
    {
        "Category": "Instruction",
        "Section": "Filling instructions",
        "Field or Concept": "Questionnaire scope",
        "Description": "Cada questionário refere-se apenas à(s) atividade(s) indicada(s), ainda que estas se desenvolvam em estabelecimentos industriais diferentes",
        "Code": None,
        "Expected Value Type": "instruction",
        "Source Location": "PDF page 4 — Instruções de preenchimento"
    },
    {
        "Category": "Instruction",
        "Section": "Filling instructions",
        "Field or Concept": "Unidade monetária",
        "Description": "Os valores monetários devem ser expressos em euros sem indicar os decimais",
        "Code": None,
        "Expected Value Type": "instruction",
        "Source Location": "PDF page 4 — Unidade monetária"
    },
    {
        "Category": "Instruction",
        "Section": "Filling instructions",
        "Field or Concept": "Arredondamentos",
        "Description": "Arredondar por excesso quando as décimas forem iguais ou superiores a 5 e por defeito quando forem inferiores",
        "Code": None,
        "Expected Value Type": "instruction",
        "Source Location": "PDF page 4 — Unidade monetária"
    },
    {
        "Category": "Instruction",
        "Section": "Filling instructions",
        "Field or Concept": "Exemplo de arredondamento",
        "Description": "6370,65 euros deve ser inscrito como 6371",
        "Code": None,
        "Expected Value Type": "example",
        "Source Location": "PDF page 4 — Unidade monetária"
    },
    {
        "Category": "Instruction",
        "Section": "Explanatory notes",
        "Field or Concept": "Empresa",
        "Description": "Definition of a company as a legal entity and organisational production unit with decision autonomy",
        "Code": None,
        "Expected Value Type": "definition",
        "Source Location": "PDF page 4 — Notas explicativas"
    },
    {
        "Category": "Instruction",
        "Section": "Explanatory notes",
        "Field or Concept": "Unidade de Atividade Económica (UAE)",
        "Description": "Definition of a UAE as the parts of a company contributing to one activity with a uniform manufacturing process and homogeneous products",
        "Code": None,
        "Expected Value Type": "definition",
        "Source Location": "PDF page 4 — Notas explicativas"
    },
    {
        "Category": "Instruction",
        "Section": "Explanatory notes",
        "Field or Concept": "Produtos",
        "Description": "Product designations and reference units correspond to the Portuguese PRODCOM list under Regulation nº 3924/91, adapted to the 2015 production survey",
        "Code": None,
        "Expected Value Type": "definition",
        "Source Location": "PDF page 4 — Notas explicativas"
    },
    {
        "Category": "Instruction",
        "Section": "Explanatory notes",
        "Field or Concept": "Quantidades produzidas",
        "Description": "Definition of quantities produced in the UAE with own raw materials, for intraconsumption in another UAE and under work-on-behalf-of-others arrangements",
        "Code": None,
        "Expected Value Type": "definition",
        "Source Location": "PDF page 4 — Notas explicativas"
    },
    {
        "Category": "Instruction",
        "Section": "Explanatory notes",
        "Field or Concept": "Quantidades vendidas",
        "Description": "Definition of quantities sold during the month, including finished and intermediate products, by-products and waste, with stated inclusions and exclusions",
        "Code": None,
        "Expected Value Type": "definition",
        "Source Location": "PDF page 4 — Notas explicativas"
    },
    {
        "Category": "Instruction",
        "Section": "Explanatory notes",
        "Field or Concept": "Valor das vendas / prestação de serviços",
        "Description": "Definitions of sales value and industrial-service value, including valuation basis, exclusions and SNC account references",
        "Code": None,
        "Expected Value Type": "definition",
        "Source Location": "PDF page 4 — Notas explicativas"
    }
]


reference_values_df = pd.DataFrame(
    reference_values,
    columns=REFERENCE_FIELDS
)

display(reference_values_df)


,Category,Section,Field or Concept,Description,Code,Expected Value Type,Source Location
0,Instrument metadata,Header,Survey name,IMPI - Inquérito Mensal à Produção Industrial,None,text,PDF page 1 — Header
1,Instrument metadata,Header,Statistical system,Instrumento de notação do Sistema Estatístico ...,None,text,PDF page 1 — Legal notice
2,Instrument metadata,Header,Legal basis,Lei nº 22/2008 de 13 de Maio; resposta confide...,None,text,PDF page 1 — Legal notice
3,Instrument metadata,Header,INE registration number,Registado no INE sob o nº 10067,None,identifier,PDF page 1 — Legal notice
4,Instrument metadata,Header,Validity date,Válido até 2026/12/31,None,date,PDF page 1 — Legal notice
...,...,...,...,...,...,...,...
64,Instruction,Explanatory notes,Unidade de Atividade Económica (UAE),Definition of a UAE as the parts of a company ...,None,definition,PDF page 4 — Notas explicativas
65,Instruction,Explanatory notes,Produtos,Product designations and reference units corre...,None,definition,PDF page 4 — Notas explicativas
66,Instruction,Explanatory notes,Quantidades produzidas,Definition of quantities produced in the UAE w...,None,definition,PDF page 4 — Notas explicativas
67,Instruction,Explanatory notes,Quantidades vendidas,Definition of quantities sold during the month...,None,definition,PDF page 4 — Notas explicativas


In [30]:
# ============================================================
# 10. Reference schema, counts and field types
# ============================================================

reference_schema_valid = (
    list(reference_values_df.columns)
    == REFERENCE_FIELDS
)

reference_record_count_valid = (
    len(reference_values_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

observed_category_counts = (
    reference_values_df["Category"]
    .value_counts()
    .to_dict()
)

reference_category_counts_valid = (
    observed_category_counts
    == EXPECTED_CATEGORY_COUNTS
)

categories_valid = set(
    reference_values_df["Category"]
) == ALLOWED_CATEGORIES

mandatory_columns = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Expected Value Type",
    "Source Location"
]

mandatory_fields_complete = not (
    reference_values_df[
        mandatory_columns
    ]
    .isna()
    .any()
    .any()
)

string_type_issues = []

for row_index, row in reference_values_df.iterrows():
    for field in mandatory_columns:
        if not isinstance(
            row[field],
            str
        ):
            string_type_issues.append(
                {
                    "row_index": int(row_index),
                    "field": field,
                    "observed_type": type(
                        row[field]
                    ).__name__
                }
            )

    code_value = row["Code"]

    if (
        code_value is not None
        and not isinstance(
            code_value,
            str
        )
    ):
        string_type_issues.append(
            {
                "row_index": int(row_index),
                "field": "Code",
                "observed_type": type(
                    code_value
                ).__name__
            }
        )


field_types_valid = (
    len(string_type_issues) == 0
)

duplicate_record_count = int(
    reference_values_df.duplicated(
        subset=REFERENCE_FIELDS,
        keep=False
    ).sum()
)

duplicate_records_absent = (
    duplicate_record_count == 0
)

reference_validation_passed = all(
    [
        reference_schema_valid,
        reference_record_count_valid,
        reference_category_counts_valid,
        categories_valid,
        mandatory_fields_complete,
        field_types_valid,
        duplicate_records_absent
    ]
)




In [31]:
# ============================================================
# 11. UAE template and form-structure integrity checks
# ============================================================

expected_uae_template_labels = {
    "Código da UAE",
    "Designação da UAE",
    "Situação da UAE perante a atividade",
    "Observações da UAE",
    "Confirmar",
    "Produtos"
}

observed_uae_template_labels = set(
    reference_values_df.loc[
        reference_values_df["Category"]
        == "UAE template element",
        "Field or Concept"
    ]
)

uae_template_valid = (
    observed_uae_template_labels
    == expected_uae_template_labels
)


reference_period_field_valid = (
    (
        reference_values_df["Field or Concept"]
        == "Referência dos dados"
    ).sum()
    == 1
)


expected_product_table_labels = {
    "NIF",
    "UAE",
    "Período de Referência",
    "Nº",
    "Produto",
    "Unid.",
    "Código",
    "Quantidades produzidas",
    "Quantidades vendidas",
    "Valor das vendas / prestação de serviços",
    "Observações empresa",
    "Observações INE"
}

observed_product_table_labels = set(
    reference_values_df.loc[
        reference_values_df["Category"]
        == "Product table field",
        "Field or Concept"
    ]
)

product_table_structure_valid = (
    observed_product_table_labels
    == expected_product_table_labels
)


print(
    "UAE template valid:",
    uae_template_valid
)

print(
    "Reference-period field valid:",
    reference_period_field_valid
)

print(
    "Product-table structure valid:",
    product_table_structure_valid
)


if not uae_template_valid:
    raise AssertionError(
        "D10 UAE template reference structure is invalid."
    )

if not reference_period_field_valid:
    raise AssertionError(
        "D10 Referência dos dados field is missing or duplicated."
    )

if not product_table_structure_valid:
    raise AssertionError(
        "D10 product-table reference structure is invalid."
    )

source_location_pattern_valid = bool(
    reference_values_df[
        "Source Location"
    ].str.match(
        r"^PDF page [1-4] — .+$"
    ).all()
)

print(
    "Source-location pattern valid:",
    source_location_pattern_valid
)

if not source_location_pattern_valid:
    raise AssertionError(
        "One or more D10 source locations have an invalid format."
    )

uae_template_valid = bool(uae_template_valid)
reference_period_field_valid = bool(reference_period_field_valid)
product_table_structure_valid = bool(product_table_structure_valid)
source_location_pattern_valid = bool(source_location_pattern_valid)

reference_schema_valid = bool(reference_schema_valid)
reference_record_count_valid = bool(reference_record_count_valid)
reference_category_counts_valid = bool(reference_category_counts_valid)
categories_valid = bool(categories_valid)
field_types_valid = bool(field_types_valid)
mandatory_fields_complete = bool(mandatory_fields_complete)

PAGE_COUNT_VALID = bool(PAGE_COUNT_VALID)
TEXT_EXTRACTABLE = bool(TEXT_EXTRACTABLE)
OCR_REQUIRED = bool(OCR_REQUIRED)

duplicate_record_count = int(duplicate_record_count)

REFERENCE_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "source_format": SOURCE_FORMAT,

    "expected_page_count": int(EXPECTED_PAGE_COUNT),
    "observed_page_count": int(PAGE_COUNT),
    "page_count_valid": bool(PAGE_COUNT_VALID),

    "text_extractable": bool(TEXT_EXTRACTABLE),
    "ocr_required": bool(OCR_REQUIRED),

    "reference_schema_valid":
        bool(reference_schema_valid),

    "record_count_valid":
        bool(reference_record_count_valid),

    "category_counts_valid":
        bool(reference_category_counts_valid),

    "categories_valid":
        bool(categories_valid),

    "field_types_valid":
        bool(field_types_valid),

    "mandatory_fields_complete":
        bool(mandatory_fields_complete),

    "duplicate_record_count":
        int(duplicate_record_count),

    "source_location_pattern_valid":
        bool(source_location_pattern_valid),

    "uae_template_valid":
        bool(uae_template_valid),

    "reference_period_field_valid":
        bool(reference_period_field_valid),

    "product_table_structure_valid":
        bool(product_table_structure_valid),

    "manual_reference_construction": True,
    "manual_structural_annotation": True,
    "calculation_applied": False,
    "semantic_inference_applied": False,
    "unit_conversion_applied": False,
    "source_value_repair_applied": False,
    "reference_values_branch_independent": True,
}

REFERENCE_INTEGRITY["reference_integrity_passed"] = bool(
    all([
        REFERENCE_INTEGRITY["page_count_valid"],
        REFERENCE_INTEGRITY["text_extractable"],
        REFERENCE_INTEGRITY["reference_schema_valid"],
        REFERENCE_INTEGRITY["record_count_valid"],
        REFERENCE_INTEGRITY["category_counts_valid"],
        REFERENCE_INTEGRITY["categories_valid"],
        REFERENCE_INTEGRITY["field_types_valid"],
        REFERENCE_INTEGRITY["mandatory_fields_complete"],
        REFERENCE_INTEGRITY["duplicate_record_count"] == 0,
        REFERENCE_INTEGRITY["source_location_pattern_valid"],
        REFERENCE_INTEGRITY["uae_template_valid"],
        REFERENCE_INTEGRITY["reference_period_field_valid"],
        REFERENCE_INTEGRITY["product_table_structure_valid"]
    ])
)

print(
    json.dumps(
        REFERENCE_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)

UAE template valid: True
Reference-period field valid: True
Product-table structure valid: True
Source-location pattern valid: True
{
  "document_id": "D10",
  "source_file": "D10 - IMPI_QUESTIONARIO INE Portugal 2026 (1).pdf",
  "source_file_sha256": "fb72aac548f61578bc9ba52448793b81bf61e37eff146f9519124d0794a4f4e5",
  "source_format": "PDF",
  "expected_page_count": 4,
  "observed_page_count": 4,
  "page_count_valid": true,
  "text_extractable": true,
  "ocr_required": false,
  "reference_schema_valid": true,
  "record_count_valid": true,
  "category_counts_valid": true,
  "categories_valid": true,
  "field_types_valid": true,
  "mandatory_fields_complete": true,
  "duplicate_record_count": 0,
  "source_location_pattern_valid": true,
  "uae_template_valid": true,
  "reference_period_field_valid": true,
  "product_table_structure_valid": true,
  "manual_reference_construction": true,
  "manual_structural_annotation": true,
  "calculation_applied": false,
  "semantic_inference_applied"

In [32]:
# ============================================================
# 12. Document and page-level characterisation
# ============================================================

page_marker_rules = {
    1: {
        "Identification section":
            r"Identificação da unidade estatística",
        "Activity-status section":
            r"Situação da unidade estatística",
        "Observations section":
            r"III\s+Observações",
        "Responsible-person section":
            r"Responsável pelo preenchimento"
    },
    2: {
        "UAE definition":
            r"Unidade de Atividade Económica",
        "Repeated UAE blocks":
            r"Situação da UAE perante a atividade",
        "Product action":
            r"Produtos"
    },
    3: {
        "Product table":
            r"QUANTIDADES\s+PRODUZIDAS",
        "Sales-value column":
            r"VALOR DAS VENDAS",
        "INE observations":
            r"OBSERVAÇÕES\s+INE"
    },
    4: {
        "Filling instructions":
            r"INSTRUÇÕES DE PREENCHIMENTO",
        "Explanatory notes":
            r"NOTAS EXPLICATIVAS",
        "Monetary rules":
            r"Unidade monetária",
        "PRODCOM":
            r"PRODCOM"
    }
}


page_feature_rows = []

for page in page_texts:
    page_number = page["page_number"]
    text = page["text"]

    marker_status = {
        marker: bool(
            re.search(
                pattern,
                text,
                flags=re.IGNORECASE
            )
        )
        for marker, pattern
        in page_marker_rules[
            page_number
        ].items()
    }

    page_feature_rows.append(
        {
            "Page Number": page_number,
            "Numeric Token Count": len(
                re.findall(
                    r"(?<!\w)-?\d+(?:[.,]\d+)?",
                    text
                )
            ),
            "Questionnaire Code Count": len(
                re.findall(
                    r"\bBC\d{3}\b",
                    text
                )
            ),
            "Contains Dense Form Layout":
                page_number in [1, 2, 3],
            "Contains Grid Table":
                page_number == 3,
            "Contains Long Instructions":
                page_number == 4,
            "Marker Status":
                json.dumps(
                    marker_status,
                    ensure_ascii=False
                ),
            "All Expected Page Markers Present":
                all(marker_status.values())
        }
    )


page_features_df = pd.DataFrame(
    page_feature_rows
)

page_characterisation_df = (
    page_characterisation_df
    .merge(
        page_features_df,
        on="Page Number",
        how="left"
    )
)


DOCUMENT_CHARACTERISATION = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "source_format": SOURCE_FORMAT,

    "page_count": PAGE_COUNT,
    "page_count_valid": PAGE_COUNT_VALID,

    "text_extractable": TEXT_EXTRACTABLE,
    "ocr_required": OCR_REQUIRED,

    "total_character_count": len(FULL_TEXT),
    "total_word_count": len(FULL_TEXT.split()),

    # Observable structural characteristics
    "contains_form_layout": True,
    "contains_blank_response_fields": True,
    "contains_checkboxes": True,
    "contains_date_fields": True,
    "contains_printed_field_codes": True,
    "contains_repeated_uae_blocks": True,
    "contains_product_entry_grid": True,
    "contains_explanatory_instructions": True,
    "contains_multicolumn_layout": True,
    "contains_dense_numeric_table": False,

    "represented_sections": [
        "Instrument identification and reference data",
        "I — Identificação da unidade estatística",
        "II — Situação da unidade estatística no período de referência dos dados",
        "III — Observações",
        "IV — Responsável pelo preenchimento",
        "Unidade de Atividade Económica (UAE)",
        "Product reporting table",
        "Instruções de preenchimento",
        "Notas explicativas"
    ],

    "fixed_extraction_scope": (
        "Instrument metadata, questionnaire fields, reusable UAE "
        "template elements, product-table fields and explanatory "
        "instructions explicitly represented in the supplied "
        "four-page IMPI questionnaire."
    ),

    "excluded_from_reference_scope": [
        "Blank respondent answers",
        "Unobserved company-specific values",
        "Repeated instances of the same UAE template element",
        "Sample product rows as respondent observations",
        "Values requiring calculation or inference",
        "Derived information not explicitly represented in the source"
    ],

    "reference_schema_version": "v1",
    "reference_task_version": "v1",

}


DOCUMENT_CHARACTERISATION_PATH.write_text(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

display(page_characterisation_df)

print(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        ensure_ascii=False,
        indent=2
    )
)


,Page Number,Text Characters,Word Count,Non-empty Lines,Text Block Count,Drawing Count,Embedded Image Count,Interactive Widget Count,Width,Height,Orientation,Numeric Token Count,Questionnaire Code Count,Contains Dense Form Layout,Contains Grid Table,Contains Long Instructions,Marker Status,All Expected Page Markers Present
0,1,1633,242,69,36,215,2,0,595.200012,841.679993,portrait,16,8,True,False,False,"{""Identification section"": true, ""Activity-sta...",True
1,2,748,111,19,13,54,1,0,595.200012,841.679993,portrait,0,0,True,False,False,"{""UAE definition"": true, ""Repeated UAE blocks""...",True
2,3,327,49,31,16,1044,0,0,595.200012,841.679993,portrait,8,0,True,True,False,"{""Product table"": true, ""Sales-value column"": ...",True
3,4,3965,587,53,15,25,0,0,595.200012,841.679993,portrait,14,0,False,False,True,"{""Filling instructions"": true, ""Explanatory no...",True


{
  "document_id": "D10",
  "document_name": "IMPI — Inquérito Mensal à Produção Industrial",
  "source_file": "D10 - IMPI_QUESTIONARIO INE Portugal 2026 (1).pdf",
  "source_file_sha256": "fb72aac548f61578bc9ba52448793b81bf61e37eff146f9519124d0794a4f4e5",
  "source_format": "PDF",
  "page_count": 4,
  "page_count_valid": true,
  "text_extractable": true,
  "ocr_required": false,
  "total_character_count": 6676,
  "total_word_count": 989,
  "contains_form_layout": true,
  "contains_blank_response_fields": true,
  "contains_checkboxes": true,
  "contains_date_fields": true,
  "contains_printed_field_codes": true,
  "contains_repeated_uae_blocks": true,
  "contains_product_entry_grid": true,
  "contains_explanatory_instructions": true,
  "contains_multicolumn_layout": true,
  "contains_dense_numeric_table": false,
  "represented_sections": [
    "Instrument identification and reference data",
    "I — Identificação da unidade estatística",
    "II — Situação da unidade estatística no perí

In [33]:
# ============================================================
# 13. Indicator-level document assessment
# ============================================================

indicator_assessment = [
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Reading Order Quality",
        "Score": "Medium",
        "Evidence Source":
            "PDF text extraction + rendered-page inspection",
        "Justification":
            "The document contains extractable text, but page 1 uses "
            "a spatial form layout and pages 2 and 3 rely on repeated "
            "blocks and grid structures whose relationships are not "
            "fully represented by linear text order."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Table Structure Integrity",
        "Score": "Medium",
        "Evidence Source":
            "Rendered-page inspection + PDF structural diagnostics",
        "Justification":
            "The product-entry grid on page 3 has clearly defined "
            "columns, but preservation of row-column associations "
            "depends on spatial layout. Page 2 also contains repeated "
            "structured UAE blocks."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Section/Header Hierarchy",
        "Score": "Low",
        "Evidence Source":
            "Manual document inspection",
        "Justification":
            "Major sections are explicitly labelled using Roman "
            "numerals, headings and named questionnaire regions, "
            "providing a clear overall hierarchy."
    },

    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Sharpness",
        "Score": "Low",
        "Evidence Source":
            "Manual visual inspection",
        "Justification":
            "The supplied questionnaire is visually clear and the "
            "relevant labels, codes, table headings and instructions "
            "are sharply represented."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Noise / Degradation",
        "Score": "Low",
        "Evidence Source":
            "Manual visual inspection",
        "Justification":
            "No meaningful scanning noise, blur or visual degradation "
            "affects the relevant document content."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "OCR Dependency",
        "Score": "Low",
        "Evidence Source":
            "Automated PDF text extraction",
        "Justification":
            "The document contains a usable native text layer and the "
            "relevant textual content can be extracted without OCR."
    },

    {
        "Dimension": "Semantic Quality",
        "Indicator": "Terminology Consistency",
        "Score": "Low",
        "Evidence Source":
            "Manual source inspection",
        "Justification":
            "Questionnaire terminology is stable across the document, "
            "including recurring concepts such as UAE, quantities "
            "produced, quantities sold and sales value."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Schema Alignment",
        "Score": "Medium",
        "Evidence Source":
            "Reference-schema comparison",
        "Justification":
            "The extraction schema must represent several structurally "
            "different element types, including instrument metadata, "
            "blank questionnaire fields, UAE template controls, product "
            "table fields and explanatory instructions."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Numerical Density",
        "Score": "Low",
        "Evidence Source":
            "Automated text profiling + manual inspection",
        "Justification":
            "Although the questionnaire contains numeric response "
            "fields, codes and an example monetary value, the supplied "
            "blank form is dominated by labels, field structures and "
            "instructions rather than observed numerical data."
    },

    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Required Field Presence",
        "Score": "Low",
        "Evidence Source":
            "Reference-value verification",
        "Justification":
            "All elements required by the predefined fixed extraction "
            "scope are represented in the supplied four-page document."
    },
    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Internal Consistency",
        "Score": "Low",
        "Evidence Source":
            "Manual source and reference inspection",
        "Justification":
            "Labels, questionnaire codes, UAE structures, product "
            "reporting fields and explanatory definitions are "
            "internally coherent across the supplied document."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Format Heterogeneity",
        "Score": "High",
        "Evidence Source":
            "Document profiling + rendered-page inspection",
        "Justification":
            "The document combines spatial form fields, checkboxes, "
            "date boxes, printed field codes, repeated UAE templates, "
            "a large product-entry grid and long narrative instructions."
    },
    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Unit / Label Variability",
        "Score": "Medium",
        "Evidence Source":
            "Reference and source inspection",
        "Justification":
            "The questionnaire combines identifiers, dates, CAE codes, "
            "checkbox responses, quantities, product units, monetary "
            "values, contact fields and several specialised labels, "
            "while still following a controlled questionnaire vocabulary."
    }
]


indicator_assessment_df = pd.DataFrame(
    indicator_assessment
)

display(
    indicator_assessment_df
)

,Dimension,Indicator,Score,Evidence Source,Justification
0,Structural Readiness,Reading Order Quality,Medium,PDF text extraction + rendered-page inspection,"The document contains extractable text, but pa..."
1,Structural Readiness,Table Structure Integrity,Medium,Rendered-page inspection + PDF structural diag...,The product-entry grid on page 3 has clearly d...
2,Structural Readiness,Section/Header Hierarchy,Low,Manual document inspection,Major sections are explicitly labelled using R...
3,Visual/OCR Readiness,Sharpness,Low,Manual visual inspection,The supplied questionnaire is visually clear a...
4,Visual/OCR Readiness,Noise / Degradation,Low,Manual visual inspection,"No meaningful scanning noise, blur or visual d..."
5,Visual/OCR Readiness,OCR Dependency,Low,Automated PDF text extraction,The document contains a usable native text lay...
6,Semantic Quality,Terminology Consistency,Low,Manual source inspection,Questionnaire terminology is stable across the...
7,Semantic Quality,Schema Alignment,Medium,Reference-schema comparison,The extraction schema must represent several s...
8,Semantic Quality,Numerical Density,Low,Automated text profiling + manual inspection,Although the questionnaire contains numeric re...
9,Completeness and Consistency,Required Field Presence,Low,Reference-value verification,All elements required by the predefined fixed ...


In [34]:
# ============================================================
# 14. Validate indicator assessment
# ============================================================

VALID_SCORES = {
    "Low",
    "Medium",
    "High"
}

expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}

invalid_scores = (
    set(
        indicator_assessment_df[
            "Score"
        ].dropna().unique()
    )
    - VALID_SCORES
)

observed_indicators = set(
    indicator_assessment_df[
        "Indicator"
    ]
)

missing_indicators = (
    expected_indicators
    - observed_indicators
)

unexpected_indicators = (
    observed_indicators
    - expected_indicators
)

if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores: {invalid_scores}"
    )

if missing_indicators:
    raise ValueError(
        f"Missing indicators: {missing_indicators}"
    )

if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators: {unexpected_indicators}"
    )

if len(indicator_assessment_df) != len(expected_indicators):
    raise ValueError(
        "Duplicate indicator rows detected."
    )

print(
    "Indicator assessment validation passed."
)

Indicator assessment validation passed.


In [35]:
# ============================================================
# 15. Dimension-level assessment
# ============================================================

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

indicator_assessment_df[
    "Numeric Score"
] = indicator_assessment_df[
    "Score"
].map(score_to_numeric)


dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),
        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)


def classify_dimension_score(mean_score):

    if mean_score < 1.5:
        return "Low"

    elif mean_score < 2.5:
        return "Medium"

    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = dimension_assessment_df[
    "Mean_Score"
].apply(
    classify_dimension_score
)

dimension_assessment_df[
    "Mean_Score"
] = dimension_assessment_df[
    "Mean_Score"
].round(2)

display(
    dimension_assessment_df
)

,Dimension,Mean_Score,Number_of_Indicators,Dimension Score
0,Completeness and Consistency,1.00,2,Low
1,Representation and Normalisation Complexity,2.50,2,High
2,Semantic Quality,1.33,3,Low
3,Structural Readiness,1.67,3,Medium
4,Visual/OCR Readiness,1.00,3,Low


In [36]:
# ============================================================
# 16. Structured quality-assessment evidence
# ============================================================

QUALITY_EVIDENCE = {
    "document_id":
        DOCUMENT_ID,

    "assessment_basis":
        "Observed document evidence was mapped to the "
        "predefined Low, Medium, and High operational "
        "criteria defined in Table 3.3 of the methodology.",

    "evidence_method":
        "Evidence was obtained through automated profiling "
        "where measurable characteristics could be derived "
        "programmatically and through documented manual "
        "inspection where qualitative assessment was required.",

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },

        "aggregation":
            "Arithmetic mean of indicator scores within "
            "each dimension.",

        "classification_rule": {
            "Low":
                "mean < 1.5",

            "Medium":
                "1.5 <= mean < 2.5",

            "High":
                "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}

In [37]:
# ============================================================
# 17. Reference summary and experiment metadata
# ============================================================

REFERENCE_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,

    "reference_record_count":
        int(len(reference_values_df)),

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "record_count_valid":
        reference_record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_valid":
        reference_category_counts_valid,

    "source_pages_represented": [
        1, 2, 3, 4
    ],

    "fields":
        REFERENCE_FIELDS
}


REFERENCE_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "source_format":
        SOURCE_FORMAT,

    "reference_file":
        "D10_reference_values.csv",

    "reference_construction_method":
        (
            "Manual document-grounded structural "
            "annotation and verification"
        ),

    "expected_reference_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_record_count":
        int(
            len(reference_values_df)
        ),

    "reference_fields":
        REFERENCE_FIELDS,

    "blank_form_fields_represented_as":
        "Expected structural elements rather than respondent values",

    "repeated_uae_blocks_deduplicated":
        True,

    "sample_product_rows_treated_as_observations":
        False,

    "manual_calculation_applied":
        False,

    "semantic_inference_applied":
        False,

    "unit_conversion_applied":
        False,

    "source_value_repair_applied":
        False,

    "reference_values_branch_independent":
        True,

    "reference_values_to_be_reused_for_branches": [
        "A",
        "B",
        "C"
    ],

    "notes":
        (
            "D10 is a blank structured questionnaire. Reference "
            "observations therefore represent document-grounded "
            "fields, template elements, table fields, metadata and "
            "instructions rather than unobserved respondent answers."
        )
}


REFERENCE_SUMMARY_PATH.write_text(
    json.dumps(
        REFERENCE_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

print(
    json.dumps(
        REFERENCE_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D10",
  "document_name": "IMPI — Inquérito Mensal à Produção Industrial",
  "reference_record_count": 69,
  "expected_record_count": 69,
  "record_count_valid": true,
  "expected_category_counts": {
    "Instrument metadata": 8,
    "Questionnaire field": 32,
    "UAE template element": 6,
    "Product table field": 12,
    "Instruction": 11
  },
  "observed_category_counts": {
    "Questionnaire field": 32,
    "Product table field": 12,
    "Instruction": 11,
    "Instrument metadata": 8,
    "UAE template element": 6
  },
  "category_counts_valid": true,
  "source_pages_represented": [
    1,
    2,
    3,
    4
  ],
  "fields": [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Code",
    "Expected Value Type",
    "Source Location"
  ]
}


In [38]:
# ============================================================
# 18. Export outputs
# ============================================================

reference_values_df.to_csv(
    REFERENCE_VALUES_PATH,
    index=False,
    encoding="utf-8-sig"
)


REFERENCE_VALUES_JSON_PATH.write_text(
    json.dumps(
        reference_values,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)


REFERENCE_SCHEMA_PATH.write_text(
    json.dumps(
        REFERENCE_SCHEMA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)


EXTRACTION_SCHEMA_PATH.write_text(
    json.dumps(
        EXTRACTION_SCHEMA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)


EXTRACTION_TASK_PATH.write_text(
    EXTRACTION_TASK.strip(),
    encoding="utf-8",
    newline="\n"
)


REFERENCE_SUMMARY_PATH.write_text(
    json.dumps(
        REFERENCE_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)


REFERENCE_METADATA_PATH.write_text(
    json.dumps(
        REFERENCE_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)


REFERENCE_INTEGRITY_PATH.write_text(
    json.dumps(
        REFERENCE_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)


DOCUMENT_CHARACTERISATION_PATH.write_text(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)


QUALITY_EVIDENCE_PATH.write_text(
    json.dumps(
        QUALITY_EVIDENCE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)


page_characterisation_df.to_csv(
    PAGE_CHARACTERISATION_PATH,
    index=False,
    encoding="utf-8-sig"
)


indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    INDICATOR_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)


dimension_assessment_df.to_csv(
    DIMENSION_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)


SOURCE_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        SOURCE_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)


print(
    "D10 Stage 1 artefacts exported."
)

D10 Stage 1 artefacts exported.
